In [7]:
import os
import json
from pathlib import Path
 
import torch
import torchvision
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torch.amp import GradScaler, autocast
from torchvision import transforms
from PIL import Image
import timm

from dataset import PlantWildDataset

In [8]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")

2.10.0+cu128
True
NVIDIA GeForce RTX 3060 Laptop GPU


In [ ]:
class MobileViT(nn.Module):
    """
    Full MobileViT fine-tune — all layers trainable.
    Uses a lower LR for the backbone vs the head to avoid
    overwriting pretrained weights too aggressively.
    """
 
    def __init__(self, num_classes: int,
                 model_name: str, dropout: float = 0.2):
        super().__init__()
 
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,
            global_pool="avg",
        )
        self.embed_dim = self.backbone.num_features
 
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.embed_dim, num_classes),
        )

        for param in self.backbone.parameters():
            param.requires_grad = True
 
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Model           : {model_name}")
        print(f"Total params    : {total / 1e6:.2f}M")
        print(f"Trainable params: {trainable / 1e6:.2f}M  "
              f"({100 * trainable / total:.1f}% unfrozen)")
 
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(images))
 
    def get_encoder(self):
        return self.backbone
 
    def get_param_groups(self, backbone_lr: float, head_lr: float):
        """
        Separate LRs for backbone and head.
        Backbone gets a lower LR to preserve pretrained weights.
        """
        return [
            {"params": self.backbone.parameters(), "lr": backbone_lr},
            {"params": self.head.parameters(),     "lr": head_lr},
        ]

In [10]:
def get_transforms(img_size: int = 320, train: bool = True):
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3,
                                   saturation=0.3, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    return transforms.Compose([
        transforms.Resize(int(img_size * 1.15)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

In [ ]:
def train(model, train_loader, test_loader,
          epochs, backbone_lr, head_lr, device, save_dir,
          class_weights=None):
 
    os.makedirs(save_dir, exist_ok=True)
 
    model = model.to(device)
 
    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device) if class_weights is not None else None,
        label_smoothing=0.1,
    )
 
    optimizer = torch.optim.AdamW(
        model.get_param_groups(backbone_lr, head_lr),
        weight_decay=1e-4,
    )
 
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2
    )
 
    scaler   = GradScaler("cuda")
    best_acc = 0.0
 
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
 
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
 
            optimizer.zero_grad()
            with autocast("cuda"):
                loss = criterion(model(images), labels)
 
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
 
            train_loss += loss.item()
 
        scheduler.step(epoch)  
 
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                with autocast("cuda"):
                    preds = model(images).argmax(1)
                correct += (preds == labels).sum().item()
                total   += labels.size(0)
 
        acc = 100 * correct / total
 
        print(f"Epoch {epoch:>3}/{epochs}  "
              f"Loss: {train_loss / len(train_loader):.4f}  "
              f"Test Acc: {acc:.2f}%")
 
        if acc > best_acc:
            best_acc = acc
            torch.save(
                model.get_encoder().state_dict(),
                os.path.join(save_dir, "best_image_encoder.pt")
            )
            print(f"  ✓ Best encoder saved ({best_acc:.2f}%)")
 
    print(f"\nFinetuning complete — best Test Acc: {best_acc:.2f}%")
    return model

In [ ]:
MODEL_NAME = "mobilevitv2_150"  
IMAGES_DIR  = "./data/images/plantwild"
IMG_SIZE    = 320      
BATCH_SIZE  = 16        
EPOCHS      = 20        
BACKBONE_LR = 1e-5      
HEAD_LR     = 1e-3      
SAVE_DIR    = "./checkpoints"
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")

In [13]:
train_ds = PlantWildDataset(IMAGES_DIR,
                            transform=get_transforms(IMG_SIZE, train=True),
                            split="train")
test_ds  = PlantWildDataset(IMAGES_DIR,
                            transform=get_transforms(IMG_SIZE, train=False),
                            split="test", label_map=train_ds.label_map)

train_ds.save_label_map("./data/label_map.json")

class_counts  = train_ds.get_class_counts()
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()
print(f"Class weights computed for {len(class_weights)} classes")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=4, pin_memory=True)

model = MobileViT(
    num_classes=len(train_ds.classes),
    model_name=MODEL_NAME,
    dropout=0.2,
)

model = train(
    model, train_loader, test_loader,
    epochs=EPOCHS,
    backbone_lr=BACKBONE_LR,
    head_lr=HEAD_LR,
    device=DEVICE,
    save_dir=SAVE_DIR,
    class_weights=class_weights,
)

Device: cuda
PlantWildDataset | split=train | 89 classes | 14832 images
PlantWildDataset | split=test | 89 classes | 3708 images
Label map saved → ./data/label_map.json
Class weights computed for 89 classes
Model           : mobilevitv2_150
Total params    : 9.89M
Trainable params: 9.89M  (100.0% unfrozen)
Epoch   1/20  Loss: 3.3929  Test Acc: 55.74%
  ✓ Best encoder saved (55.74%)
Epoch   2/20  Loss: 2.5140  Test Acc: 60.81%
  ✓ Best encoder saved (60.81%)
Epoch   3/20  Loss: 2.3051  Test Acc: 64.16%
  ✓ Best encoder saved (64.16%)
Epoch   4/20  Loss: 2.1911  Test Acc: 65.10%
  ✓ Best encoder saved (65.10%)
Epoch   5/20  Loss: 2.1258  Test Acc: 66.26%
  ✓ Best encoder saved (66.26%)
Epoch   6/20  Loss: 2.0619  Test Acc: 67.31%
  ✓ Best encoder saved (67.31%)
Epoch   7/20  Loss: 2.0176  Test Acc: 67.48%
  ✓ Best encoder saved (67.48%)
Epoch   8/20  Loss: 1.9957  Test Acc: 67.29%
Epoch   9/20  Loss: 1.9704  Test Acc: 67.48%
Epoch  10/20  Loss: 1.9592  Test Acc: 67.88%
  ✓ Best encoder s